# CorePromoter model energy vs. natural promoter conservation (E. coli)

這個 notebook 拿 `Model_CorePromoter_clean.ipynb` 訓練出來的 CorePromoter model，
把它學到的「各位置鹼基 energy」跟真實 E. coli promoter 的位置保守度做關聯分析。

## 為什麼可以這樣比

`CorePromoterModel` 的 `conv2` 權重是**固定**的：它只是把 `conv1` 的 8 個 8-bp filter
擺在固定位移上相加（3 個 channel 對應 spacer 16/17/18）。所以整個模型的 energy 是
嚴格可加的，可以無損拆成一個 4 × L 的 **energy matrix** `e(i, b)`：

```
E(sequence) = Σ_i e(i, base_i) + conv2.bias[channel]
```

## 座標系統：以 −10 為基準

相對座標 0 = −10 hexamer 的第一個鹼基。`conv2` 把 −35 filter 固定在 position 22，
所以以 −10 對齊時 UP 與 −35 會隨 spacer 平移，spacer 下游則不動。
本分析**只用 spacer = 17**（E. coli 最大宗），對應 conv2 channel 1：

| 元件 | 相對座標 |
|---|---|
| UP (filter 0, 1) | −42 … −27 |
| −35 (filter 2) | −23 … −16 |
| spacer (filter 3, 4) | −14 … +1 |
| −10 3' 半段 + discriminator (filter 5) | +2 … +9 |
| ITS / downstream (filter 6, 7) | +10 … +25 |

8 個 filter × 8 bp = **64 個位置**（−42…+25 之間有 4 個未覆蓋的空隙：−26、−25、−24、−15）。

### kernel 編號與 spacer 自由度

程式裡的 `filter` 欄是 **0-based**（`f0`…`f7`）；口語上的「第 N 個 kernel」是 1-based，
兩者差 1。以下用 1-based 敘述。

conv2 的 tap 位置在三個 channel 之間只有一處不同：

| kernel | #1 | #2 | #3 (−35) | #4 | #5 | #6 | #7 | #8 |
|---|---|---|---|---|---|---|---|---|
| tap (sp16) | 3 | 11 | 22 | 30 | 38 | 46 | 54 | 62 |
| tap (sp17) | 3 | 11 | 22 | 31 | 39 | 47 | 55 | 63 |
| tap (sp18) | 3 | 11 | 22 | 32 | 40 | 48 | 56 | 64 |

**#1–#3 固定，#4–#8 整組同步滑動**，所以唯一隨 spacer 改變的空隙在 **#3 → #4 之間**
（0 / 1 / 2 bp）。#2 → #3 之間另有一個固定的 3 bp 空隙，與 spacer 無關。

因為那個可變空隙在 #5 的**上游**，#4 以後的 kernel 相對 −10 的 register 是**固定的**：
−10 的第一個鹼基永遠落在 **kernel #5 的第 7 格**（0-based `k=6`），三個 spacer 都一樣。
初始化 motif 也是這樣排的：`ATGGGG|TA` + `TAAT|TTTT` → `TA` + `TAAT` = **TATAAT**。
第 5 節結尾有一段 assert 會用天然共識序列自動驗證這個錨點。

## 兩個保守度指標

對每個相對位置 i，從對齊後的真實 promoter 算出鹼基頻率 `f_b(i)`：

- **Information content**（均勻背景）：`IC(i) = 2 − H(i)`，`H(i) = −Σ_b f_b log2 f_b`
- **KL divergence**（基因體背景 `p_b`）：`KL(i) = Σ_b f_b log2(f_b / p_b)`

`p_b` 由 `genomes/NC_000913.2.gb` 全基因體鹼基組成算得。

> **注意**：E. coli 的基因體組成非常接近均勻（GC 50.8%），
> 所以 `KL(i) ≈ IC(i)`，兩者差距只有 ~0.01 bits 等級，圖 1 與圖 2 會長得幾乎一樣。
> 這個設計要延伸到 49 物種時才會顯出差別（例如 *Streptomyces* GC ~72%、*Mycoplasma* GC ~25%）。
> 若想讓 KL 在 E. coli 也有鑑別力，把 `BACKGROUND_SOURCE` 改成 `"flank"`
> （改用 promoter 上游側翼序列當背景）。

## 資料來源

`tables/Data_S1_20250826.xlsx` 的 `Es.co`（1865 筆，有基因註解）與
`Es.co$`（4357 筆，完整 TSS 清單）兩張 sheet 都算，互相對照。

表格裡 `UP + Minus35 + Spacer + Minus10 + Dis + Start + ITR` 串起來就是**連續的基因體序列**
（抽驗 300 筆，跨接點的 120 bp 探針有 98% / 96.7% 能在基因體中原樣找到），
所以不需要用座標回查基因體。

> 表格座標**對不上** `genomes/NC_000913.2.gb`（`Es.co$` 標的是 NC_000913.**3**，
> 兩版之間有 indel 差異，−10 座標比對只有 ~50% / ~38% 命中）。
> 因此基因體檔案在這裡**只用來算背景鹼基組成**，序列一律取自表格。

## 輸出

**總覽圖**（五個元件畫在同一格，帶整體相關係數）

- 圖 1／圖 2：一個點 = 一個位置。x = 該位置 4 個鹼基 energy 的 max−min（區辨力），y = IC / KL
- 圖 3／圖 4：一個點 = (位置, 鹼基)。x = 中心化 energy，y = logo letter height / per-base KL 貢獻

**拆解圖**（`_byElement_`，2 列 sheet × 5 欄元件，每格獨立的相關係數與 bootstrap CI）

- 圖 5–8：與圖 1–4 一一對應，用來比較模型對各元件學到的能量與天然保守度的關聯強弱差異

**表**：`outputs/` 下的 per-position、per-base 明細表，以及含 per-element 分項的相關係數彙總表

In [ ]:
## Setup
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from Bio import SeqIO
from scipy.stats import pearsonr, spearmanr

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "recursive_corepromoter_design.py").exists():
    for base in [SCRIPT_DIR, *SCRIPT_DIR.parents]:
        cand = base / "MS2_Data_PyTorch" / "scripts"
        if (cand / "recursive_corepromoter_design.py").exists():
            SCRIPT_DIR = cand
            break
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import recursive_corepromoter_design as legacy

MS2_DIR = legacy.MS2_DIR
TABLE_DIR = legacy.TABLE_DIR
WEIGHTS_DIR = legacy.WEIGHTS_DIR
GENOME_DIR = MS2_DIR / "genomes"
FIG_DIR = MS2_DIR / "figures"
OUT_DIR = MS2_DIR / "outputs"
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

DATA_S1 = TABLE_DIR / "Data_S1_20250826.xlsx"
CHECKPOINT = WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
GENOME_GB = GENOME_DIR / "NC_000913.2.gb"  # Escherichia coli K-12 MG1655

# --- analysis settings ---
SPACER = 17                     # 只分析 spacer=17（conv2 channel 1）
SHEETS = ["Es.co", "Es.co$"]    # 兩張 E. coli sheet 都算並比較
BACKGROUND_SOURCE = "genome"    # "genome" = 全基因體組成; "flank" = promoter 上游側翼組成
FLANK_BG_WIDTH = 50             # BACKGROUND_SOURCE="flank" 時取 UP 欄位最前面幾 bp
CORE_RANGE = (-23, 9)           # 相關係數彙總表裡「core promoter only」子集的相對座標範圍

# --- model settings ---
RETRAIN = False                 # True = 依 clean notebook 流程重訓（需要 tables 裡的 PL/SL/DL/UL/ITS pkl）
RETRAIN_EPOCHS = 50

BASES = ["A", "C", "G", "T"]
BASE_COLORS = {"A": "#2ca02c", "C": "#1f77b4", "G": "#ff7f0e", "T": "#d62728"}
ELEMENT_ORDER = ["UP", "-35", "spacer", "-10", "DIS/downstream"]
ELEMENT_COLORS = {
    "UP": "#937860",
    "-35": "#4c72b0",
    "spacer": "#b8b8b8",
    "-10": "#c44e52",
    "DIS/downstream": "#dd8452",
}

M35_LEN = 6
M10_LEN = 6
M35_FILTER_IDX = 2              # conv1 filter index 2 = TTGACATT (−35)
SPACER_CHANNEL = {16: 0, 17: 1, 18: 2}

device = torch.device("cpu")
print(f"MS2_DIR   : {MS2_DIR}")
print(f"spacer    : {SPACER}  (conv2 channel {SPACER_CHANNEL[SPACER]})")
print(f"background: {BACKGROUND_SOURCE}")

## 1. 載入模型

預設直接載入 `weights/weights_CorePromoter_clean.pt`。
把 `RETRAIN` 設成 `True` 就會改用 `recursive_corepromoter_design.train_core_model()`
重跑一次 clean notebook 的訓練流程（同樣的資料組裝、同樣的 50 epoch）。

In [ ]:
def load_core_model():
    if RETRAIN:
        print(f"Retraining CorePromoter model for {RETRAIN_EPOCHS} epochs ...")
        model, history, _ = legacy.train_core_model(epochs=RETRAIN_EPOCHS, device=device)
        print(f"final train/test MSE (log10): "
              f"{history['train_mse_log10'].iloc[-1]:.4f} / {history['test_mse_log10'].iloc[-1]:.4f}")
        return model, {"source": "retrained in-notebook", "epochs": RETRAIN_EPOCHS}

    ckpt = legacy.torch_load_weights(CHECKPOINT, device)
    model = legacy.CorePromoterModel(
        seq_length=int(ckpt["seq_length"]),
        num_conds=int(ckpt["num_conds"]),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    meta = {k: v for k, v in ckpt.items() if not hasattr(v, "shape") and k != "model_state_dict"}
    print(f"Loaded {CHECKPOINT.name}")
    for k, v in meta.items():
        print(f"  {k}: {v}")
    return model.eval(), meta


model, model_meta = load_core_model()
model.eval()
print(f"\nconv1.weight {tuple(model.conv1.weight.shape)}  (filter, base, position)")
print(f"conv2.weight {tuple(model.conv2.weight.shape)}  (channel, filter, position)")

## 2. 把模型拆成以 −10 為基準的 energy matrix

從 `conv2` 讀出每個 filter 的固定位移，換算成相對 −10 的座標，
再把 `conv1.weight[filter, base, k]` 攤平成 `e(rel_pos, base)`。

In [ ]:
def conv2_filter_offsets(core_model, channel):
    """每個 conv1 filter 在 conv2 kernel 上的固定位移。"""
    w2 = core_model.conv2.weight.detach().cpu().numpy()[channel]  # (n_filters, 65)
    offsets = []
    for f in range(w2.shape[0]):
        nz = np.flatnonzero(np.abs(w2[f]) > 1e-6)
        if nz.size != 1:
            raise ValueError(f"conv2 channel {channel} filter {f} has {nz.size} taps, expected 1")
        offsets.append(int(nz[0]))
    return offsets


def annotate_element(rel, spacer):
    if 0 <= rel < M10_LEN:
        return "-10"
    if -spacer <= rel < 0:
        return "spacer"
    if -(spacer + M35_LEN) <= rel < -spacer:
        return "-35"
    if rel < -(spacer + M35_LEN):
        return "UP"
    return "DIS/downstream"


def build_energy_matrix(core_model, spacer):
    """回傳以 −10 hexamer 第一個鹼基為 rel_pos=0 的 energy matrix。"""
    channel = SPACER_CHANNEL[spacer]
    offsets = conv2_filter_offsets(core_model, channel)
    m10_start = offsets[M35_FILTER_IDX] + M35_LEN + spacer
    w1 = core_model.conv1.weight.detach().cpu().numpy()  # (n_filters, 4, k)

    rows = []
    for f, off in enumerate(offsets):
        for k in range(w1.shape[2]):
            rows.append({
                "rel_pos": off + k - m10_start,
                "filter": f,
                "filter_pos": k,
                **{b: float(w1[f, bi, k]) for bi, b in enumerate(BASES)},
            })

    em = pd.DataFrame(rows).sort_values("rel_pos").reset_index(drop=True)
    if not em["rel_pos"].is_unique:
        raise ValueError("conv2 filter footprints overlap; energy matrix is not additive per position")

    em["element"] = [annotate_element(r, spacer) for r in em["rel_pos"]]
    em["energy_span"] = em[BASES].max(axis=1) - em[BASES].min(axis=1)
    em["energy_sd"] = em[BASES].std(axis=1, ddof=0)
    em["energy_mean"] = em[BASES].mean(axis=1)
    return em, offsets, m10_start


energy_mat, filter_offsets, model_m10_start = build_energy_matrix(model, SPACER)

print(f"conv2 filter offsets (channel {SPACER_CHANNEL[SPACER]}): {filter_offsets}")
print(f"−35 start = {filter_offsets[M35_FILTER_IDX]}, −10 start = {model_m10_start}")
print(f"modelled positions: {len(energy_mat)}  "
      f"(rel {energy_mat['rel_pos'].min()} … {energy_mat['rel_pos'].max()})")
missing = sorted(set(range(energy_mat["rel_pos"].min(), energy_mat["rel_pos"].max() + 1))
                 - set(energy_mat["rel_pos"]))
print(f"uncovered gaps: {missing}")
print()
print(energy_mat.groupby("element", sort=False).agg(
    n_pos=("rel_pos", "size"),
    rel_min=("rel_pos", "min"),
    rel_max=("rel_pos", "max"),
    mean_span=("energy_span", "mean"),
).reindex([e for e in ELEMENT_ORDER if e in set(energy_mat["element"])]).round(3))

## 3. 讀入真實 E. coli promoter 並以 −10 對齊

In [ ]:
FIELDS = ["UP", "Minus35", "Spacer", "Minus10", "Dis", "Start", "ITR"]


def load_promoters(sheet, spacer):
    raw = pd.read_excel(DATA_S1, sheet_name=sheet)
    n_raw = len(raw)

    d = raw.dropna(subset=FIELDS).copy()
    n_complete = len(d)
    for f in FIELDS:
        d[f] = d[f].astype(str).str.upper().str.strip()

    d = d[d["Spacer"].str.len() == spacer]
    d = d[(d["Minus35"].str.len() == M35_LEN) & (d["Minus10"].str.len() == M10_LEN)]

    d["aligned"] = d[FIELDS].agg("".join, axis=1)
    d["m10_index"] = d["UP"].str.len() + M35_LEN + spacer

    stats = {
        "sheet": sheet,
        "n_raw": n_raw,
        "n_complete": n_complete,
        "n_spacer": len(d),
    }
    return d.reset_index(drop=True), stats


def position_counts(promoters, rel_positions):
    """對齊後每個相對位置的 A/C/G/T 計數。"""
    rel_min, rel_max = int(min(rel_positions)), int(max(rel_positions))
    width = rel_max - rel_min + 1

    windows = []
    for seq, m10 in zip(promoters["aligned"].to_numpy(), promoters["m10_index"].to_numpy()):
        start, stop = int(m10) + rel_min, int(m10) + rel_max + 1
        if start >= 0 and stop <= len(seq):
            windows.append(seq[start:stop])
    if not windows:
        raise ValueError("no promoter is long enough to cover the requested window")

    arr = np.array(windows, dtype=f"S{width}").view("S1").reshape(len(windows), width)
    counts = np.stack([(arr == b.encode()).sum(axis=0) for b in BASES], axis=1)

    out = pd.DataFrame(counts, columns=BASES)
    out.insert(0, "rel_pos", np.arange(rel_min, rel_max + 1))
    out["n_acgt"] = out[BASES].sum(axis=1)
    return out, len(windows)


rel_positions = energy_mat["rel_pos"].to_numpy()
promoter_sets, count_tables, load_stats = {}, {}, []

for sheet in SHEETS:
    prom, stats = load_promoters(sheet, SPACER)
    counts, n_used = position_counts(prom, rel_positions)
    stats["n_windowed"] = n_used
    promoter_sets[sheet] = prom
    count_tables[sheet] = counts
    load_stats.append(stats)

load_stats = pd.DataFrame(load_stats)
load_stats.columns = ["sheet", "rows in sheet", "no missing element", f"spacer={SPACER}", "in alignment window"]
print(load_stats.to_string(index=False))

## 4. 背景鹼基組成

In [ ]:
def genome_background(gb_path):
    record = next(SeqIO.parse(gb_path, "genbank"))
    seq = str(record.seq).upper()
    counts = np.array([seq.count(b) for b in BASES], dtype=float)
    return counts / counts.sum(), record


def flank_background(promoters, width):
    seqs = promoters["UP"].str[:width]
    joined = "".join(seqs)
    counts = np.array([joined.count(b) for b in BASES], dtype=float)
    return counts / counts.sum()


genome_bg, genome_record = genome_background(GENOME_GB)
print(f"{genome_record.id}  {genome_record.description}")
print(f"  length {len(genome_record.seq):,} bp   GC {genome_bg[1] + genome_bg[2]:.4f}")
print("  genome background: " + "  ".join(f"{b}={p:.5f}" for b, p in zip(BASES, genome_bg)))

if BACKGROUND_SOURCE == "genome":
    background = genome_bg
elif BACKGROUND_SOURCE == "flank":
    background = flank_background(promoter_sets[SHEETS[0]], FLANK_BG_WIDTH)
    print(f"\n  flank background (UP[:{FLANK_BG_WIDTH}] of {SHEETS[0]}): "
          + "  ".join(f"{b}={p:.5f}" for b, p in zip(BASES, background)))
else:
    raise ValueError(f"unknown BACKGROUND_SOURCE: {BACKGROUND_SOURCE!r}")

## 5. 每個位置的 information content 與 KL divergence

In [ ]:
def conservation_table(counts, background):
    freq = counts[BASES].to_numpy(dtype=float)
    freq = freq / freq.sum(axis=1, keepdims=True)

    log_f = np.zeros_like(freq)
    np.log2(freq, out=log_f, where=freq > 0)      # f = 0 的位置留 0，f·log f 也是 0
    entropy = -(freq * log_f).sum(axis=1)
    kl = (freq * (log_f - np.log2(background)[None, :])).sum(axis=1)

    out = pd.DataFrame({
        "rel_pos": counts["rel_pos"].to_numpy(),
        "n_acgt": counts["n_acgt"].to_numpy(),
        "entropy_bits": entropy,
        "IC_bits": 2.0 - entropy,
        "KL_bits": kl,
    })
    for i, b in enumerate(BASES):
        out[f"freq_{b}"] = freq[:, i]
    return out


per_position = {}
for sheet in SHEETS:
    cons = conservation_table(count_tables[sheet], background)
    merged = energy_mat.merge(cons, on="rel_pos", how="left", validate="one_to_one")
    merged.insert(0, "sheet", sheet)
    per_position[sheet] = merged

per_position_df = pd.concat(per_position.values(), ignore_index=True)

print("max |KL − IC| per sheet (E. coli 背景接近均勻，兩者幾乎相同):")
for sheet in SHEETS:
    d = per_position[sheet]
    print(f"  {sheet:8s} {np.abs(d['KL_bits'] - d['IC_bits']).max():.4f} bits")

print(f"\ntop conserved positions ({SHEETS[0]}):")
print(per_position[SHEETS[0]]
      .nlargest(10, "IC_bits")[["rel_pos", "element", "IC_bits", "KL_bits", "energy_span"]]
      .round(3).to_string(index=False))


# --- 錨點自我驗證 ---
# −10 起點的定義是 m10_start = kernel #3 (−35 filter) 的 conv2 tap + M35_LEN + spacer。
# 只要座標錯位 1 bp，天然共識序列就讀不出 TATAAT / TTGACA，下面的 assert 會擋下來。
def consensus_seq(position_df, lo, hi):
    idx = position_df.set_index("rel_pos")
    return "".join(
        BASES[int(np.argmax([idx.at[r, f"freq_{b}"] for b in BASES]))]
        for r in range(lo, hi + 1)
    )


print("\nanchor self-check (每個 sheet 都必須讀出 TTGACA / TATAAT)")
for sheet in SHEETS:
    m35_seq = consensus_seq(per_position[sheet], -(SPACER + M35_LEN), -(SPACER + 1))
    m10_seq = consensus_seq(per_position[sheet], 0, M10_LEN - 1)
    print(f"  {sheet:8s} rel {-(SPACER + M35_LEN)}..{-(SPACER + 1)} = {m35_seq}   rel 0..{M10_LEN - 1} = {m10_seq}")
    assert m35_seq == "TTGACA", f"{sheet}: −35 錨點錯位，讀到 {m35_seq}"
    assert m10_seq == "TATAAT", f"{sheet}: −10 錨點錯位，讀到 {m10_seq}"

## 6. 每個 (位置, 鹼基) 的 logo letter height 與 per-base KL 貢獻

- logo letter height：`f_b(i) · IC(i)`
- per-base KL 貢獻：`f_b(i) · log2(f_b(i) / p_b)`，加總即為 `KL(i)`
- 模型端用**中心化** energy `e(i,b) − mean_b e(i,b)`，去掉每個位置的常數位移

In [ ]:
def build_per_base(position_df, background):
    records = []
    bg = dict(zip(BASES, background))
    for _, row in position_df.iterrows():
        energies = np.array([row[b] for b in BASES], dtype=float)
        centered = energies - energies.mean()
        for i, b in enumerate(BASES):
            f = float(row[f"freq_{b}"])
            records.append({
                "sheet": row["sheet"],
                "rel_pos": int(row["rel_pos"]),
                "element": row["element"],
                "base": b,
                "energy": float(energies[i]),
                "energy_centered": float(centered[i]),
                "freq": f,
                "letter_height": f * float(row["IC_bits"]),
                "kl_contrib": f * np.log2(f / bg[b]) if f > 0 else 0.0,
            })
    return pd.DataFrame(records)


per_base = {sheet: build_per_base(per_position[sheet], background) for sheet in SHEETS}
per_base_df = pd.concat(per_base.values(), ignore_index=True)

print(per_base_df.groupby("sheet").agg(
    n=("rel_pos", "size"),
    energy_centered_min=("energy_centered", "min"),
    energy_centered_max=("energy_centered", "max"),
    letter_height_max=("letter_height", "max"),
    kl_contrib_min=("kl_contrib", "min"),
    kl_contrib_max=("kl_contrib", "max"),
).round(3))

## 7. 相關係數彙總

In [ ]:
N_BOOT = 5000        # bootstrap 重抽樣次數
BOOT_SEED = 0


def _pearson_rows(X, Y):
    """一次算完 (n_boot, n) 重抽樣矩陣每一列的 Pearson r；退化樣本回傳 nan。"""
    Xc = X - X.mean(axis=1, keepdims=True)
    Yc = Y - Y.mean(axis=1, keepdims=True)
    num = (Xc * Yc).sum(axis=1)
    den = np.sqrt((Xc ** 2).sum(axis=1) * (Yc ** 2).sum(axis=1))
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)


def bootstrap_r_ci(x, y, n_boot=N_BOOT, seed=BOOT_SEED, alpha=0.05):
    """Pearson r 的 bootstrap 百分位信賴區間。n 很小時退化的重抽樣會被濾掉。"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    rs = _pearson_rows(x[idx], y[idx])
    rs = rs[np.isfinite(rs)]
    if rs.size < 100:                     # 有效重抽樣太少，CI 沒有意義
        return np.nan, np.nan
    lo, hi = np.percentile(rs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def correlate(x, y, n_boot=0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    pr, pp = pearsonr(x, y)
    sr = spearmanr(x, y)
    out = {"n": int(ok.sum()), "pearson_r": pr, "pearson_p": pp,
           "spearman_rho": float(sr.correlation), "spearman_p": float(sr.pvalue)}
    if n_boot:
        out["r_lo"], out["r_hi"] = bootstrap_r_ci(x, y, n_boot=n_boot)
    return out


# 四個 (層級, x, y) 組合，後面的彙總與繪圖共用
METRIC_SPECS = [
    ("per position", "energy_span", "IC_bits"),
    ("per position", "energy_span", "KL_bits"),
    ("per (position, base)", "energy_centered", "letter_height"),
    ("per (position, base)", "energy_centered", "kl_contrib"),
]


def _frame_for(sheet, level):
    return per_position[sheet] if level == "per position" else per_base[sheet]


rows = []
for sheet in SHEETS:
    # 既有的整體與 core 子集
    for subset_name, mask in [("all positions", None),
                              (f"core {CORE_RANGE[0]}..{CORE_RANGE[1]}", CORE_RANGE)]:
        for level, xcol, ycol in METRIC_SPECS:
            d = _frame_for(sheet, level)
            if mask is not None:
                d = d[d["rel_pos"].between(*mask)]
            rows.append({"sheet": sheet, "subset": subset_name, "level": level,
                         "x": xcol, "y": ycol, **correlate(d[xcol], d[ycol])})

    # 新增：逐元件
    for element in ELEMENT_ORDER:
        for level, xcol, ycol in METRIC_SPECS:
            d = _frame_for(sheet, level)
            d = d[d["element"] == element]
            if len(d) < 3:
                continue
            rows.append({"sheet": sheet, "subset": f"element: {element}", "level": level,
                         "x": xcol, "y": ycol, **correlate(d[xcol], d[ycol], n_boot=N_BOOT)})

correlation_df = pd.DataFrame(rows)
print(correlation_df.round(4).to_string(index=False))

## 8. 圖 1／圖 2：一個點 = 一個位置

In [ ]:
def annotate_stats(ax, x, y, loc="upper left"):
    st = correlate(x, y)
    ax.text(
        0.03 if loc == "upper left" else 0.62, 0.97,
        f"n = {st['n']}\nPearson r = {st['pearson_r']:.3f}\nSpearman ρ = {st['spearman_rho']:.3f}",
        transform=ax.transAxes, va="top", ha="left", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.8", alpha=0.85),
    )
    return st


def add_trendline(ax, x, y, color="black"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    xs = np.linspace(x[ok].min(), x[ok].max(), 50)
    ax.plot(xs, slope * xs + intercept, color=color, lw=1, ls="--", zorder=1)


def plot_per_position(metric, ylabel, filename, n_annotate=5):
    fig, axes = plt.subplots(1, len(SHEETS), figsize=(5.4 * len(SHEETS), 4.6),
                             dpi=150, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, sheet in zip(axes, SHEETS):
        d = per_position[sheet]
        for element in ELEMENT_ORDER:
            sub = d[d["element"] == element]
            if sub.empty:
                continue
            ax.scatter(sub["energy_span"], sub[metric], s=42, alpha=0.9,
                       color=ELEMENT_COLORS[element], edgecolors="white", linewidths=0.6,
                       label=element, zorder=3)
        add_trendline(ax, d["energy_span"], d[metric])
        annotate_stats(ax, d["energy_span"], d[metric])

        for _, r in d.nlargest(n_annotate, metric).iterrows():
            rel = int(r["rel_pos"])
            ax.annotate("0" if rel == 0 else f"{rel:+d}", (r["energy_span"], r[metric]),
                        textcoords="offset points", xytext=(6, 4), fontsize=7, color="0.25")

        ax.set_title(f"{sheet}  (n = {len(promoter_sets[sheet]):,} promoters)", fontsize=10)
        ax.set_xlabel("model energy span at position\n" + r"$\max_b\,e(i,b)-\min_b\,e(i,b)$")
        ax.axhline(0, color="0.85", lw=0.8, zorder=0)

    axes[0].set_ylabel(ylabel)
    axes[-1].legend(frameon=False, fontsize=8, title="element", title_fontsize=8,
                    loc="lower right")
    fig.suptitle(f"E. coli promoters, spacer = {SPACER}, aligned on −10   |   "
                 f"one point = one position (n = {len(energy_mat)})", fontsize=10)
    fig.tight_layout()
    for ext in ("svg", "png"):
        fig.savefig(FIG_DIR / f"{filename}.{ext}", bbox_inches="tight")
    plt.show()
    return fig


plot_per_position("IC_bits", "Information content (bits)",
                  f"EnergyVsIC_position_Ecoli_sp{SPACER}")

In [ ]:
plot_per_position("KL_bits", f"KL divergence vs {BACKGROUND_SOURCE} background (bits)",
                  f"EnergyVsKL_position_Ecoli_sp{SPACER}")

## 9. 圖 3／圖 4：一個點 = (位置, 鹼基)

> **怎麼讀圖 3**：logo letter height `f_b·IC` 恆為非負，所以**強偏好**與**強排斥**的鹼基
> 會落在同一側（都接近 0 或都不接近 0），對 energy 是非單調的 —— 被排斥的鹼基 `f_b ≈ 0`
> → letter height ≈ 0 但 energy 很負；中性鹼基 `f_b ≈ 0.25`、`IC ≈ 0`
> → letter height ≈ 0 且 energy ≈ 0。因此 Pearson 為正但 Spearman 幾乎是 0。
>
> 圖 4 的 per-base KL 貢獻**帶正負號**（排斥的鹼基為負），才是 energy matrix 真正對應的量，
> Pearson 與 Spearman 會一致地呈現正相關。

In [ ]:
def plot_per_base(metric, ylabel, filename):
    fig, axes = plt.subplots(1, len(SHEETS), figsize=(5.4 * len(SHEETS), 4.6),
                             dpi=150, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for ax, sheet in zip(axes, SHEETS):
        d = per_base[sheet]
        for b in BASES:
            sub = d[d["base"] == b]
            ax.scatter(sub["energy_centered"], sub[metric], s=26, alpha=0.75,
                       color=BASE_COLORS[b], edgecolors="none", label=b, zorder=3)
        add_trendline(ax, d["energy_centered"], d[metric])
        annotate_stats(ax, d["energy_centered"], d[metric])

        ax.axhline(0, color="0.85", lw=0.8, zorder=0)
        ax.axvline(0, color="0.85", lw=0.8, zorder=0)
        ax.set_title(f"{sheet}  (n = {len(promoter_sets[sheet]):,} promoters)", fontsize=10)
        ax.set_xlabel("centered model energy\n" + r"$e(i,b)-\overline{e(i,\cdot)}$")

    axes[0].set_ylabel(ylabel)
    axes[-1].legend(frameon=False, fontsize=8, title="base", title_fontsize=8, loc="lower right")
    fig.suptitle(f"E. coli promoters, spacer = {SPACER}, aligned on −10   |   "
                 f"one point = (position, base) (n = {len(energy_mat) * 4})", fontsize=10)
    fig.tight_layout()
    for ext in ("svg", "png"):
        fig.savefig(FIG_DIR / f"{filename}.{ext}", bbox_inches="tight")
    plt.show()
    return fig


plot_per_base("letter_height", "Logo letter height  " + r"$f_b(i)\cdot IC(i)$  (bits)",
              f"EnergyVsIC_perbase_Ecoli_sp{SPACER}")

In [ ]:
plot_per_base("kl_contrib", "Per-base KL contribution  " + r"$f_b(i)\,\log_2(f_b(i)/p_b)$  (bits)",
              f"EnergyVsKL_perbase_Ecoli_sp{SPACER}")

## 10. 圖 5–8：依元件拆解

前面四張圖把 64 個位置（或 256 個 (位置, 鹼基)）擠在同一格，各元件的點大量重疊，
而且 y 軸範圍差 20 倍以上（−10 的 IC 到 1.60，UP 只到 0.045），看不出模型對**哪個元件**
學到的能量最貼近天然保守度。以下把每張圖拆成 2 列（sheet）× 5 欄（元件）。

- **x 軸全部共用**，方便比較各元件的 energy 區辨力落在什麼範圍
- **y 軸依欄獨立**（`sharey="col"`）：同一元件的兩張 sheet 共用刻度，不同元件各自縮放，
  所以**跨欄比較 y 的絕對高度沒有意義**，要看統計框裡的數字
- 統計框標 `n / r [bootstrap 95% CI] / ρ / p`

> **小樣本警告**：per-position 層級的 −35 與 −10 各只有 **6 個位置**。
> r 看起來很高（~0.92–0.96）且 p 顯著，但 n=6 的 bootstrap CI 很寬，
> 不要把這兩格的點估計當成精確值 —— 它說明的是「方向明確」，不是「強度精確」。

**預期會看到的梯度**：−10 ≈ −35 > spacer > UP ≫ DIS/downstream。
其中 DIS/downstream 在 per-position 層級甚至是**負相關** —— 模型在該區有明顯的能量區辨
（energy span 最高到 0.96），但天然 E. coli 在那裡幾乎沒有保守性（IC 最高只有 0.067）。
這代表模型從合成 library 學到的 discriminator / ITS 偏好，在天然 promoter 裡沒有對應的
選擇壓力痕跡。

圖 7（letter height 版本）的 spacer 與 UP 兩格 Spearman 會特別低，原因就是第 9 節說的
非單調性：這兩個元件大多數位置 `IC ≈ 0`，letter height 全部擠在 0 附近，排序資訊被洗掉。

> **UP 在兩個層級的落差**：UP 在 per-position 層級只有 r ≈ 0.27–0.34（CI 跨過 0），
> 到了 per-base KL 層級卻跳到 r ≈ 0.61（CI [0.48, 0.72]）。原因是 UP 區各位置的 IC 都極小
> （最高 0.045 bits），位置層級的保守度基本上是雜訊，排不出高低；但**鹼基層級的方向性**還在
> —— 模型的 UP filter 偏好 A/T，天然 UP 區也是 AT-rich，所以帶正負號的 per-base KL 貢獻
> 仍然對得上。也就是說模型抓到的是 UP 的**組成偏好**，而不是特定位置的保守性。

In [ ]:
def _stats_box(ax, x, y, n_boot=N_BOOT):
    st = correlate(x, y, n_boot=n_boot)
    ci = ""
    if np.isfinite(st.get("r_lo", np.nan)):
        ci = f"\n   95% CI [{st['r_lo']:+.2f}, {st['r_hi']:+.2f}]"
    ax.text(
        0.04, 0.96,
        f"n = {st['n']}\nr = {st['pearson_r']:+.3f}{ci}\nρ = {st['spearman_rho']:+.3f}"
        f"\np = {st['pearson_p']:.1e}",
        transform=ax.transAxes, va="top", ha="left", fontsize=6.2, linespacing=1.35,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.88),
        zorder=5,
    )
    return st


def plot_by_element(level, metric, ylabel, filename):
    """2 列 (sheet) x 5 欄 (element) 的拆解圖。level: 'position' | 'perbase'"""
    xcol = "energy_span" if level == "position" else "energy_centered"
    xlabel = ("model energy span at position\n" + r"$\max_b\,e(i,b)-\min_b\,e(i,b)$"
              if level == "position" else
              "centered model energy\n" + r"$e(i,b)-\overline{e(i,\cdot)}$")

    fig, axes = plt.subplots(
        len(SHEETS), len(ELEMENT_ORDER), figsize=(2.75 * len(ELEMENT_ORDER), 3.15 * len(SHEETS)),
        dpi=150, sharex="all", sharey="col", squeeze=False,
    )

    for row, sheet in enumerate(SHEETS):
        source = per_position[sheet] if level == "position" else per_base[sheet]
        for col, element in enumerate(ELEMENT_ORDER):
            ax = axes[row][col]
            d = source[source["element"] == element]

            if level == "position":
                ax.scatter(d[xcol], d[metric], s=34, alpha=0.9,
                           color=ELEMENT_COLORS[element], edgecolors="white", linewidths=0.5,
                           zorder=3)
            else:
                for b in BASES:
                    sub = d[d["base"] == b]
                    ax.scatter(sub[xcol], sub[metric], s=20, alpha=0.75,
                               color=BASE_COLORS[b], edgecolors="none", label=b, zorder=3)
                ax.axvline(0, color="0.88", lw=0.8, zorder=0)

            add_trendline(ax, d[xcol], d[metric])
            _stats_box(ax, d[xcol], d[metric])
            ax.axhline(0, color="0.88", lw=0.8, zorder=0)
            ax.tick_params(labelsize=7)

            if level == "position":
                # 點少的時候把所有位置標出來，點多就只標 y 最大的兩個
                marked = d if len(d) <= 8 else d.nlargest(2, metric)
                for _, r in marked.iterrows():
                    rel = int(r["rel_pos"])
                    ax.annotate("0" if rel == 0 else f"{rel:+d}", (r[xcol], r[metric]),
                                textcoords="offset points", xytext=(5, 3), fontsize=6,
                                color="0.25", zorder=4)

            if row == 0:
                ax.set_title(f"{element}\n{len(d)} points", fontsize=9,
                             color=ELEMENT_COLORS[element])
            if row == len(SHEETS) - 1:
                ax.set_xlabel(xlabel, fontsize=8)
            if col == 0:
                ax.set_ylabel(f"{sheet}\n{ylabel}", fontsize=8)

    # 上方留白給統計框，避免蓋到資料點。sharey="col" 會讓同欄的另一列跟著調整。
    for col in range(len(ELEMENT_ORDER)):
        lo, hi = axes[0][col].get_ylim()
        axes[0][col].set_ylim(lo, lo + (hi - lo) * 1.45)

    if level == "perbase":
        axes[0][-1].legend(frameon=False, fontsize=7, title="base", title_fontsize=7,
                           loc="upper left", bbox_to_anchor=(1.03, 1.0), borderaxespad=0)

    point_unit = "one position" if level == "position" else "(position, base)"
    fig.suptitle(
        f"E. coli promoters, spacer = {SPACER}, aligned on −10   |   "
        f"one point = {point_unit}   |   y axis scaled per element (not comparable across columns)",
        fontsize=10,
    )
    fig.tight_layout()
    for ext in ("svg", "png"):
        fig.savefig(FIG_DIR / f"{filename}.{ext}", bbox_inches="tight")
    plt.show()
    return fig


plot_by_element("position", "IC_bits", "Information content (bits)",
                f"EnergyVsIC_position_byElement_Ecoli_sp{SPACER}")

In [ ]:
plot_by_element("position", "KL_bits", f"KL divergence (bits)",
                f"EnergyVsKL_position_byElement_Ecoli_sp{SPACER}")

In [ ]:
plot_by_element("perbase", "letter_height", "Logo letter height (bits)",
                f"EnergyVsIC_perbase_byElement_Ecoli_sp{SPACER}")

In [ ]:
plot_by_element("perbase", "kl_contrib", "Per-base KL contribution (bits)",
                f"EnergyVsKL_perbase_byElement_Ecoli_sp{SPACER}")

## 11. 匯出

In [ ]:
tag = f"Ecoli_sp{SPACER}_{BACKGROUND_SOURCE}bg"
paths = {
    "per position": OUT_DIR / f"energy_vs_conservation_{tag}_positions.csv",
    "per (position, base)": OUT_DIR / f"energy_vs_conservation_{tag}_perbase.csv",
    "correlations": OUT_DIR / f"energy_vs_conservation_{tag}_correlations.csv",
}
per_position_df.to_csv(paths["per position"], index=False)
per_base_df.to_csv(paths["per (position, base)"], index=False)
correlation_df.to_csv(paths["correlations"], index=False)

for label, p in paths.items():
    print(f"{label:22s} -> {p}")
print()
for name in sorted(FIG_DIR.glob(f"EnergyVs*_Ecoli_sp{SPACER}*.*")):
    print(f"figure                 -> {name}")